In [ ]:
# # install octave
# !sudo apt-get -qq update
# !sudo apt-get -qq install octave octave-signal liboctave-dev

# # install oct2py that compatible with colab
# import os

# from pkg_resources import get_distribution

# os.system(
#     f"pip install -qq"
#     f" ipykernel=={get_distribution('ipykernel').version}"
#     f" ipython=={get_distribution('ipython').version}"
#     f" tornado=={get_distribution('tornado').version}"
#     f" oct2py"
# )

# # install packages
# !pip install -qq matpower matpowercaseframesa

In [ ]:
import oct2py

import matpower

print(f"oct2py version: {oct2py.__version__}")
print(f"matpower version: {matpower.__version__}")

oct2py version: 5.8.0
matpower version: 8.1.0.2.2.4


In [ ]:
import os

import numpy as np
from matpowercaseframes import CaseFrames

from matpower import path_matpower, start_instance

In [ ]:
path = os.path.join(path_matpower, "data/case9.m")

In [ ]:
m = start_instance()

## Original Case

In [ ]:
cf = CaseFrames(path, load_case_engine=m)

mpc = m.runpf(cf.to_mpc(), verbose=False)
cf = CaseFrames(mpc)
cf.infer_numpy()
display(cf.branch)
display(cf.bus)
display(cf.gen)

,F_BUS,T_BUS,BR_R,BR_X,BR_B,RATE_A,RATE_B,RATE_C,TAP,SHIFT,BR_STATUS,ANGMIN,ANGMAX,PF,QF,PT,QT
branch,,,,,,,,,,,,,,,,,
1,1,4,0.0000,0.0576,0.000,250,250,250,0,0,1,-360,360,71.641021,27.045924,-71.641021,-23.923127
2,4,5,0.0170,0.0920,0.158,250,250,250,0,0,1,-360,360,30.703670,1.030006,-30.537263,-16.543365
3,5,6,0.0390,0.1700,0.358,150,150,150,0,0,1,-360,360,-59.462737,-13.456635,60.816586,-18.074836
4,3,6,0.0000,0.0586,0.000,300,300,300,0,0,1,-360,360,85.000000,-10.859709,-85.000000,14.955327
5,6,7,0.0119,0.1008,0.209,150,150,150,0,0,1,-360,360,24.183414,3.119508,-24.095417,-24.295823
6,7,8,0.0085,0.0720,0.149,250,250,250,0,0,1,-360,360,-75.904583,-10.704177,76.379866,-0.797331
7,8,2,0.0000,0.0625,0.000,250,250,250,0,0,1,-360,360,-163.000000,9.178149,163.000000,6.653660
8,8,9,0.0320,0.1610,0.306,250,250,250,0,0,1,-360,360,86.620134,-8.380817,-84.320163,-11.312751
9,9,4,0.0100,0.0850,0.176,250,250,250,0,0,1,-360,360,-40.679837,-38.687249,40.937352,22.893121


,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
1,1,3,0,0,0,0,1,1.040000,0.000000,345,1,1.1,0.9
2,2,2,0,0,0,0,1,1.025000,9.280005,345,1,1.1,0.9
3,3,2,0,0,0,0,1,1.025000,4.664751,345,1,1.1,0.9
4,4,1,0,0,0,0,1,1.025788,-2.216788,345,1,1.1,0.9
5,5,1,90,30,0,0,1,1.012654,-3.687396,345,1,1.1,0.9
6,6,1,0,0,0,0,1,1.032353,1.966716,345,1,1.1,0.9
7,7,1,100,35,0,0,1,1.015883,0.727536,345,1,1.1,0.9
8,8,1,0,0,0,0,1,1.025769,3.719701,345,1,1.1,0.9
9,9,1,125,50,0,0,1,0.995631,-3.988805,345,1,1.1,0.9


,GEN_BUS,PG,QG,QMAX,QMIN,VG,MBASE,GEN_STATUS,PMAX,PMIN,...,PC2,QC1MIN,QC1MAX,QC2MIN,QC2MAX,RAMP_AGC,RAMP_10,RAMP_30,RAMP_Q,APF
gen,,,,,,,,,,,,,,,,,,,,,
1,1,71.641021,27.045924,300,-300,1.040,100,1,250,10,...,0,0,0,0,0,0,0,0,0,0
2,2,163.000000,6.653660,300,-300,1.025,100,1,300,10,...,0,0,0,0,0,0,0,0,0,0
3,3,85.000000,-10.859709,300,-300,1.025,100,1,270,10,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# save transformer rows before dropping
transformer_index = cf.branch.index[cf.branch["BR_R"] == 0]
tr_pairs = cf.branch.loc[transformer_index, ["F_BUS", "T_BUS"]].astype(int).copy()

# drop transformers from branch table (we will reduce buses / gens next)
cf.branch.drop(transformer_index, inplace=True)

# buses still used by remaining lines
lines_index = cf.branch.index[cf.branch["BR_R"] > 0]
used_buses = set(
    np.unique(cf.branch.loc[lines_index, ["F_BUS", "T_BUS"]].values).astype(int)
)

# build adjacency graph of transformer-only connectivity
adj = {}
for _, row in tr_pairs.iterrows():
    a = int(row["F_BUS"])
    b = int(row["T_BUS"])
    adj.setdefault(a, set()).add(b)
    adj.setdefault(b, set()).add(a)

# find connected components in transformer graph
seen = set()
components = []
for node in adj:
    if node in seen:
        continue
    stack = [node]
    comp = set()
    while stack:
        n = stack.pop()
        if n in seen:
            continue
        seen.add(n)
        comp.add(n)
        for nb in adj.get(n, ()):
            if nb not in seen:
                stack.append(nb)
    components.append(comp)

# for each transformer component, pick destination bus:
# prefer a bus that is connected to remaining lines; otherwise pick min(bus_id)
bus_sum_cols = [c for c in ["PD", "QD", "GS", "BS"] if c in cf.bus.columns]
moved = []
for comp in components:
    comp = set(comp)
    dest_candidates = comp & used_buses
    if dest_candidates:
        dest = sorted(dest_candidates)[0]
    else:
        # no bus in this component stays connected to lines, pick an arbitrary survivor
        dest = min(comp)

    # move other buses in component into dest
    for src in sorted(comp):
        if src == dest:
            continue
        if src not in cf.bus.index:
            continue
        # sum per-bus numeric quantities
        for c in bus_sum_cols:
            cf.bus.loc[dest, c] = cf.bus.loc[dest, c] + cf.bus.loc[src, c]
        # if source was slack, make dest slack
        if "BUS_TYPE" in cf.bus.columns and cf.bus.loc[src, "BUS_TYPE"] == 3:
            cf.bus.loc[dest, "BUS_TYPE"] = 3
        # move generators referencing src -> dest
        if "GEN_BUS" in cf.gen.columns:
            cf.gen.loc[cf.gen["GEN_BUS"].astype(int) == int(src), "GEN_BUS"] = int(dest)
        # drop the now-moved source bus row
        cf.bus.drop(src, inplace=True)
        moved.append((int(src), int(dest)))

# show result
print("moved bus -> dest:", moved)
display(cf.branch)
display(cf.bus)
display(cf.gen)

moved bus -> dest: [(1, 4), (3, 6), (2, 8)]


,F_BUS,T_BUS,BR_R,BR_X,BR_B,RATE_A,RATE_B,RATE_C,TAP,SHIFT,BR_STATUS,ANGMIN,ANGMAX,PF,QF,PT,QT
branch,,,,,,,,,,,,,,,,,
2,4,5,0.0170,0.0920,0.158,250,250,250,0,0,1,-360,360,30.703670,1.030006,-30.537263,-16.543365
3,5,6,0.0390,0.1700,0.358,150,150,150,0,0,1,-360,360,-59.462737,-13.456635,60.816586,-18.074836
5,6,7,0.0119,0.1008,0.209,150,150,150,0,0,1,-360,360,24.183414,3.119508,-24.095417,-24.295823
6,7,8,0.0085,0.0720,0.149,250,250,250,0,0,1,-360,360,-75.904583,-10.704177,76.379866,-0.797331
8,8,9,0.0320,0.1610,0.306,250,250,250,0,0,1,-360,360,86.620134,-8.380817,-84.320163,-11.312751
9,9,4,0.0100,0.0850,0.176,250,250,250,0,0,1,-360,360,-40.679837,-38.687249,40.937352,22.893121


,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
4,4,3,0,0,0,0,1,1.025788,-2.216788,345,1,1.1,0.9
5,5,1,90,30,0,0,1,1.012654,-3.687396,345,1,1.1,0.9
6,6,1,0,0,0,0,1,1.032353,1.966716,345,1,1.1,0.9
7,7,1,100,35,0,0,1,1.015883,0.727536,345,1,1.1,0.9
8,8,1,0,0,0,0,1,1.025769,3.719701,345,1,1.1,0.9
9,9,1,125,50,0,0,1,0.995631,-3.988805,345,1,1.1,0.9


,GEN_BUS,PG,QG,QMAX,QMIN,VG,MBASE,GEN_STATUS,PMAX,PMIN,...,PC2,QC1MIN,QC1MAX,QC2MIN,QC2MAX,RAMP_AGC,RAMP_10,RAMP_30,RAMP_Q,APF
gen,,,,,,,,,,,,,,,,,,,,,
1,4,71.641021,27.045924,300,-300,1.040,100,1,250,10,...,0,0,0,0,0,0,0,0,0,0
2,8,163.000000,6.653660,300,-300,1.025,100,1,300,10,...,0,0,0,0,0,0,0,0,0,0
3,6,85.000000,-10.859709,300,-300,1.025,100,1,270,10,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
mpc = m.runpf(cf.to_mpc(), verbose=False)
cf = CaseFrames(mpc)
cf.infer_numpy()
display(cf.branch)
display(cf.bus)
display(cf.gen)

,F_BUS,T_BUS,BR_R,BR_X,BR_B,RATE_A,RATE_B,RATE_C,TAP,SHIFT,BR_STATUS,ANGMIN,ANGMAX,PF,QF,PT,QT
branch,,,,,,,,,,,,,,,,,
1,4,5,0.0170,0.0920,0.158,250,250,250,0,0,1,-360,360,30.515345,-12.933126,-30.365960,-3.335616
2,5,6,0.0390,0.1700,0.358,150,150,150,0,0,1,-360,360,-59.634040,-26.664384,60.937556,-7.780875
3,6,7,0.0119,0.1008,0.209,150,150,150,0,0,1,-360,360,24.062444,-3.078834,-23.994723,-20.381971
4,7,8,0.0085,0.0720,0.149,250,250,250,0,0,1,-360,360,-76.005277,-14.618029,76.439472,1.153576
5,8,9,0.0320,0.1610,0.306,250,250,250,0,0,1,-360,360,86.560528,5.500085,-84.349115,-28.242889
6,9,4,0.0100,0.0850,0.176,250,250,250,0,0,1,-360,360,-40.650885,-21.757111,40.823012,4.453808


,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
1,4,3,0,0,0,0,1,1.040000,-2.216788,345,1,1.1,0.9
2,5,1,90,30,0,0,1,1.039264,-3.744746,345,1,1.1,0.9
3,6,1,0,0,0,0,1,1.077811,1.301106,345,1,1.1,0.9
4,7,1,100,35,0,0,1,1.066897,0.146221,345,1,1.1,0.9
5,8,1,0,0,0,0,1,1.078292,2.846696,345,1,1.1,0.9
6,9,1,125,50,0,0,1,1.025156,-4.006756,345,1,1.1,0.9


,GEN_BUS,PG,QG,QMAX,QMIN,VG,MBASE,GEN_STATUS,PMAX,PMIN,...,PC2,QC1MIN,QC1MAX,QC2MIN,QC2MAX,RAMP_AGC,RAMP_10,RAMP_30,RAMP_Q,APF
gen,,,,,,,,,,,,,,,,,,,,,
1,4,71.338358,-8.479319,300,-300,1.040,100,1,250,10,...,0,0,0,0,0,0,0,0,0,0
2,8,163.000000,6.653660,300,-300,1.025,100,1,300,10,...,0,0,0,0,0,0,0,0,0,0
3,6,85.000000,-10.859709,300,-300,1.025,100,1,270,10,...,0,0,0,0,0,0,0,0,0,0
